In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler


DATA_PATH = "Data/crsp_ff3_daily_1995_2024.parquet"

df = pd.read_parquet(DATA_PATH)

print(df.shape)
df.head()


(14657813, 11)


,permno,date,ret,prc,vol,shrout,mktrf,smb,hml,rf,excess_ret
0,10026,1995-01-03,-0.010753,11.5,18872.0,9408.0,-0.0026,-0.0097,0.0094,0.0002,-0.010953
1,10026,1995-01-04,0.0,11.5,33900.0,9408.0,0.0032,-0.0029,0.0047,0.0002,-0.0002
2,10026,1995-01-05,-0.01087,11.375,6202.0,9408.0,-0.0005,0.0004,0.0005,0.0002,-0.01107
3,10026,1995-01-06,0.010989,11.5,7510.0,9408.0,0.0018,-0.0001,0.0013,0.0002,0.010789
4,10026,1995-01-09,-0.01087,11.375,4000.0,9408.0,0.0008,0.0002,0.0013,0.0002,-0.01107


In [ ]:
train = df[df["date"] <= "2022-12-31"].copy()
val   = df[(df["date"] >= "2023-01-01") & (df["date"] <= "2023-12-31")].copy()
test  = df[(df["date"] >= "2024-01-01") & (df["date"] <= "2024-12-31")].copy()

print("Train:", train.shape)
print("Val  :", val.shape)
print("Test :", test.shape)


Train: (12808974, 11)
Val  : (906417, 11)
Test : (942422, 11)


In [ ]:
np.random.seed(42)

all_permnos = train["permno"].unique()
sample_permnos = np.random.choice(all_permnos, size=50, replace=False)

train_sub = train[train["permno"].isin(sample_permnos)].copy()
val_sub   = val[val["permno"].isin(sample_permnos)].copy()
test_sub  = test[test["permno"].isin(sample_permnos)].copy()

print("Subsample sizes:")
print(train_sub.shape, val_sub.shape, test_sub.shape)


Subsample sizes:
(176193, 11) (12500, 11) (12560, 11)


In [ ]:
features = ["mktrf", "smb", "hml"]
target = "excess_ret"


In [ ]:
def fit_ff3(stock_train):
    X = stock_train[features]
    X = sm.add_constant(X)
    y = stock_train[target]
    model = sm.OLS(y, X).fit()
    return model

In [ ]:
ff3_models = {}

for permno in sample_permnos:
    stock_train = train_sub[train_sub["permno"] == permno]
    ff3_models[permno] = fit_ff3(stock_train)

len(ff3_models)


50

In [ ]:
def predict_ff3(model, stock_data):
    X = stock_data[features]
    X = sm.add_constant(X)
    return model.predict(X)

test_predictions = []

for permno in sample_permnos:
    model = ff3_models[permno]
    stock_test = test_sub[test_sub["permno"] == permno]
    
    preds = predict_ff3(model, stock_test)
    
    tmp = stock_test[["date", "permno", target]].copy()
    tmp["ff3_pred"] = preds.values
    test_predictions.append(tmp)

ff3_test_results = pd.concat(test_predictions)

ff3_test_results.head()

mse_ff3 = np.mean(
    (ff3_test_results[target] - ff3_test_results["ff3_pred"])**2
)

print("FF3 OOS MSE:", mse_ff3)


FF3 OOS MSE: 0.0018140727813194246


In [ ]:
class ShallowNN(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, x):
        return self.net(x)

In [ ]:
def train_nn(X_train, y_train, X_val, y_val, hidden_dim=16, lr=0.001, epochs=200):
    
    model = ShallowNN(input_dim=3, hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
    
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.float32).view(-1,1)
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_train)
        loss = loss_fn(preds, y_train)
        loss.backward()
        optimizer.step()
    
    # validation loss
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val)
        val_loss = loss_fn(val_preds, y_val).item()
    
    return model, val_loss


In [ ]:
nn_models = {}
val_losses = {}

for permno in sample_permnos:
    stock_train = train_sub[train_sub["permno"] == permno]
    stock_val   = val_sub[val_sub["permno"] == permno]

    # X scaling (train only)
    x_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(stock_train[features])
    X_val   = x_scaler.transform(stock_val[features])

    # y scaling (train only)  
    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(stock_train[[target]]).flatten()
    y_val   = y_scaler.transform(stock_val[[target]]).flatten()

    model, val_loss = train_nn(X_train, y_train, X_val, y_val, hidden_dim=16, lr=0.001, epochs=50)

    # store model + both scalers  
    nn_models[permno] = (model, x_scaler, y_scaler)
    val_losses[permno] = val_loss

print("Average validation loss (scaled y):", np.mean(list(val_losses.values())))

Average validation loss (scaled y): 0.8032224418967963


In [ ]:
nn_test_predictions = []

for permno in sample_permnos:
    model, x_scaler, y_scaler = nn_models[permno]  # <-- NEW unpack

    stock_test = test_sub[test_sub["permno"] == permno]

    X_test = x_scaler.transform(stock_test[features])
    X_test_t = torch.tensor(X_test, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test_t).numpy().flatten()

    # invert y scaling  <-- NEW
    preds = y_scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()

    tmp = stock_test[["date", "permno", target]].copy()
    tmp["nn_pred"] = preds
    nn_test_predictions.append(tmp)

nn_test_results = pd.concat(nn_test_predictions, ignore_index=True)
nn_test_results.head()


,date,permno,excess_ret,nn_pred
0,2024-01-02,17889,0.002544,-0.000390
1,2024-01-03,17889,-0.052305,0.014114
2,2024-01-04,17889,-0.036402,0.007109
3,2024-01-05,17889,0.060176,0.001836
4,2024-01-08,17889,-0.005959,0.012525


In [ ]:
mse_nn = np.mean((nn_test_results[target] - nn_test_results["nn_pred"])**2)

print("FF3 OOS MSE:", mse_ff3)
print("NN  OOS MSE:", mse_nn)


FF3 OOS MSE: 0.0018140727813194246
NN  OOS MSE: 0.0019179038884313165
